# Évaluation du modèle FinBERT — Analyse de sentiment financier

**Référence :** *Hands-On AI Trading with Python, QuantConnect, and AWS* — Section 06, Example 19.
**Projet associé :** `ML-FinBERT-Sentiment` (stratégie : `main.py`).

Ce notebook **évalue le modèle FinBERT** ([ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert)) sur des titres de presse financière. Il démontre que le **vrai modèle SOTA s'exécute localement** (PyTorch + CUDA), contrairement à QC Cloud qui ne peut pas charger les poids HuggingFace/TensorFlow en cours de backtest.

> **Verdict SOTA (règle [#3801](https://github.com/jsboige/CoursIA/issues/3801)) :** FinBERT = **SOTA-OK** pour l'inférence locale (modèle réel, `torch`+`transformers`+CUDA). Le *backtest complet* Ex19 (stratégie `main.py` sur flux Tiingo News) est **RECOVERABLE**, bloqué sur (a) un token Tiingo — fournisseur de données enregistré, action utilisateur — et (b) le daemon Docker local (`engine wedged`). Aucune solution de repli dégradée (baseline mot-clé seule) n'est commitée à la place du vrai modèle.

## 1. Configuration et chargement du modèle
FinBERT est un modèle BERT fine-tuné pour la classification de sentiment financier en **3 classes** : `positive`, `negative`, `neutral`. On charge le vrai modèle depuis HuggingFace.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | torch {torch.__version__}')


Device: cuda | torch 2.11.0+cu128


In [2]:
MODEL_NAME = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print('Labels:', model.config.id2label)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


## 2. Fonction de scoring

Comme dans la stratégie `main.py` (méthode `_finbert_sentiment`), le **score de sentiment** est la différence `positive − negative`, dans l'intervalle [-1, +1] : fortement positif = haussier, fortement négatif = baissier.

In [3]:
def finbert_sentiment(text):
    """Retourne les probabilités (pos, neg, neu) et le score = positive - negative."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    pos, neg, neu = float(probs[0]), float(probs[1]), float(probs[2])
    return {'positive': pos, 'negative': neg, 'neutral': neu, 'score': pos - neg}

# Exemple résolu : titre franchement positif
sample = finbert_sentiment('Apple beats Q3 earnings expectations, revenue surges 12%')
print(sample)


{'positive': 0.9435939788818359, 'negative': 0.028262481093406677, 'neutral': 0.028143538162112236, 'score': 0.9153314977884293}


## 3. Évaluation sur un jeu de titres financiers

Jeu de données **non trivial** (Prong B de la règle SOTA) : titres nuancés — signaux mixtes, prospectives, mots-pièges — pour exercer la capacité distinctive du modèle. Un cas dégénéré (titres univoques) rendrait FinBERT équivalent à un simple comptage de mots et ne mettrait pas le modèle en valeur.

In [4]:
headlines = [
    # Franchement positifs
    'Apple beats Q3 earnings expectations, revenue surges 12%',
    'Microsoft raises full-year guidance on strong Azure demand',
    # Franchement négatifs
    'Meta shares plunge after disappointing ad revenue outlook',
    'Tesla deliveries miss estimates amid production halts',
    # Nuancés / mixtes (le cas discriminant)
    'Amazon profits fall but AWS growth accelerates',
    'Netflix subscriber growth slows despite price cut',
    # Prospectives / prudentes
    'Analysts cautious on semiconductor sector ahead of Fed decision',
    'Traders await inflation data, markets flat',
    # Adversariaux (mots-pièges)
    'Company reports record losses but expects recovery next quarter',
    'Stock rallies on layoff announcement',
]

rows = []
for h in headlines:
    s = finbert_sentiment(h)
    labels = {'positive': s['positive'], 'negative': s['negative'], 'neutral': s['neutral']}
    pred = max(labels, key=labels.get)
    rows.append({
        'headline': h,
        'positive': round(s['positive'], 3),
        'negative': round(s['negative'], 3),
        'neutral': round(s['neutral'], 3),
        'score': round(s['score'], 3),
        'prediction': pred,
    })

df = pd.DataFrame(rows)
df


,headline,positive,negative,neutral,score,prediction
0,"Apple beats Q3 earnings expectations, revenue ...",0.944,0.028,0.028,0.915,positive
1,Microsoft raises full-year guidance on strong ...,0.870,0.080,0.051,0.790,positive
2,Meta shares plunge after disappointing ad reve...,0.008,0.968,0.024,-0.960,negative
3,Tesla deliveries miss estimates amid productio...,0.011,0.971,0.018,-0.960,negative
4,Amazon profits fall but AWS growth accelerates,0.126,0.856,0.018,-0.730,negative
5,Netflix subscriber growth slows despite price cut,0.012,0.951,0.037,-0.939,negative
6,Analysts cautious on semiconductor sector ahea...,0.049,0.649,0.302,-0.600,negative
7,"Traders await inflation data, markets flat",0.037,0.180,0.783,-0.143,neutral
8,Company reports record losses but expects reco...,0.039,0.946,0.015,-0.906,negative
9,Stock rallies on layoff announcement,0.034,0.898,0.069,-0.864,negative


## 4. Interprétation

Observations clés :

- Les titres **franchement** orientés sont classés correctement avec une forte confiance.
- Les titres **mixtes** (« *profits fall but AWS growth accelerates* ») révèlent l'avantage   de FinBERT sur un comptage naïf : le modèle **pèse les deux signaux** au lieu de se   déclencher sur un seul mot.
- Les titres **adversariaux** (« *rallies on layoff announcement* ») testent la compréhension   contextuelle — le mot « rally » est haussier hors contexte, mais l'annonce de licenciements   traduit une dynamique d'entreprise plus ambiguë.

Ce sont précisément ces cas que la stratégie Ex19 cherche à exploiter et qu'une baseline mot-clé (cf. section 5) rate.

## 5. Comparaison avec la baseline mot-clé

La stratégie `main.py` garde un repli `_keyword_sentiment` (comptage de listes positive/negative). Sur un sous-ensemble **adversarial**, le comptage se trompe là où FinBERT saisit le contexte.

In [5]:
def keyword_sentiment(text):
    """Baseline mot-clé — repli de main.py (_keyword_sentiment)."""
    positive = ['surge', 'rally', 'gain', 'profit', 'growth', 'beat', 'exceed', 'strong',
                'bullish', 'upgrade', 'outperform', 'raise', 'buy', 'positive', 'record',
                'high', 'soar', 'jump', 'climb', 'advance', 'recover', 'boost']
    negative = ['decline', 'drop', 'loss', 'fall', 'crash', 'miss', 'weak', 'bearish',
                'downgrade', 'sell', 'negative', 'concern', 'risk', 'fear', 'recession',
                'low', 'slump', 'plunge', 'tumble', 'cut', 'reduce', 'warning']
    t = text.lower()
    pos = sum(1 for w in positive if w in t)
    neg = sum(1 for w in negative if w in t)
    total = pos + neg
    return (pos - neg) / total if total else 0.0

adversarial = [
    'Amazon profits fall but AWS growth accelerates',
    'Stock rallies on layoff announcement',
    'Company reports record losses but expects recovery next quarter',
]
comp = [{
    'headline': h,
    'finbert_score': round(finbert_sentiment(h)['score'], 3),
    'keyword_score': round(keyword_sentiment(h), 3),
} for h in adversarial]
pd.DataFrame(comp)


,headline,finbert_score,keyword_score
0,Amazon profits fall but AWS growth accelerates,-0.730,0.333
1,Stock rallies on layoff announcement,-0.864,0.000
2,Company reports record losses but expects reco...,-0.906,0.333


## 6. Lien avec la stratégie de trading

Dans `main.py`, le score FinBERT est **agrégé sur une fenêtre glissante** (`sentiment_window` articles), et la stratégie prend position quand la moyenne dépasse un seuil (`sentiment_threshold = 0.2`) : **long** si > +0.2, **liquidation** si < -0.2. Ce notebook isole la **brique d'inférence** ; l'agrégation temporelle et la logique de trading vivent dans `main.py`.

## 7. Exercices

> Stubs **sans** `raise NotImplementedError` (règle C.1) : le notebook s'exécute de bout en bout même exercices non complétés. Complétez les `# TODO etudiant`.

In [6]:
# Exercice 1 — Votre propre jeu de titres
# Indice: choisissez 5 titres au sentiment délibérément mixte ou ambigu.
# Etape 1: complétez la liste mes_titres ci-dessous.
# Etape 2: exécutez l'inférence et commentez les cas où le modèle se trompe.
mes_titres = [
    # 'TODO etudiant: titre 1',
    # 'TODO etudiant: titre 2',
]
# for h in mes_titres:
#     print(h, finbert_sentiment(h))
print('Exercice 1 a completer')


Exercice 1 a completer


In [7]:
# Exercice 2 — Agrégation pondérée par la confiance
# Indice: un signal de trading peut pondérer chaque score par max(pos, neg, neu)
#         (haute confiance = poids fort ; neutre incertain = poids faible).
# Etape 1: écrivez score_pondere(titres) -> score agrégé pondéré.
# Etape 2: comparez-le à la moyenne simple (non pondérée).
def score_pondere(titres):
    # TODO etudiant
    return None
print('Exercice 2 a completer')


Exercice 2 a completer


In [8]:
# Exercice 3 — Fenêtre glissante de sentiment (comme main.py)
# Indice: sur un flux chronologique de nouvelles, maintenez la fenêtre des N derniers
#         scores et calculez la moyenne mobile du sentiment.
# Etape 1: écrivez sentiment_rolling(flux, fenetre) -> pd.Series de moyennes mobiles.
def sentiment_rolling(flux, fenetre=5):
    # TODO etudiant
    return None
print('Exercice 3 a completer')


Exercice 3 a completer


## Conclusion

- **FinBERT s'exécute localement** (SOTA-OK) : modèle réel, PyTorch + CUDA, 3 classes,   score = `positive − negative` cohérent avec `main.py`.
- Le modèle **dépasse la baseline mot-clé** sur les titres mixtes et adversariaux — c'est   la capacité distinctive que la stratégie Ex19 cherche à exploiter.
- Le **backtest complet Ex19** (stratégie `main.py` sur flux Tiingo News) nécessite un   **token Tiingo** (fournisseur de données enregistré) et le daemon Docker local — voir le   verdict SOTA en tête de notebook.

**Référence :** Hands-On AI Trading with Python, QuantConnect, and AWS — Section 06, Example 19.